# EV Charging Points Visualization

This notebook visualizes the distribution of Electric Vehicle (EV) charging stations across Spain.

In [ ]:
import polars as pl
import geopandas as gpd
import folium
from folium.plugins import MarkerCluster
import os

## 1. Load data

### 1.1 Load Charging Points data

In [ ]:
data_path = "../data/processed/charging_points.parquet"

if not os.path.exists(data_path):
    print(f"Error: Data file not found at {data_path}")
else:
    df_full = pl.read_parquet(data_path)
    print(f"Loaded {df_full.height} connector records.")
    df_full = df_full.filter(pl.col('max_power')>=100000)
    print(f"filter for ultra-fast chargers {df_full.height} connector records.")

df_full.head()


### 1.2 Group Charging points by location

In [ ]:
df_sites = df_full.group_by("site_id").agg([
    pl.col("site_name").first(),
    pl.col("latitude").first(),
    pl.col("longitude").first(),
    pl.col("connector_type").unique(),
    pl.col("connector_type").count().alias('n_chargers'),
    pl.col("max_power").unique(),
    pl.col("charging_mode").unique()
]).filter(
    (pl.col("latitude").is_not_null()) & (pl.col("longitude").is_not_null())
)

print(f"Grouped into {df_sites.height} unique charging sites.")
df_sites.head()

In [ ]:
data_path = "../data/processed/charging_stations.parquet"
charging_stations = gpd.read_parquet(data_path)
charging_stations = charging_stations.to_crs(epsg=4326)
charging_stations

### 1.3 Load Road Segments

In [ ]:
import geopandas as gpd
# Paths
processed_path = '../data/processed/integrated_road_network.parquet'

gdf_processed = gpd.read_parquet(processed_path)

if gdf_processed.crs != 4326:
    gdf_processed = gdf_processed.to_crs(epsg=4326)

print(f"Loaded {len(gdf_processed)} Road Segments.")
gdf_processed.head()

## 2. Interactive Maps

We use **CartoDB Positron** tiles for a clean visual style. Below are two ways to visualize the EV charging infrastructure:

1. **Clustered View**: Groups nearby chargers to show density at different zoom levels. This provides a better regional overview.
2. **Individual Points View**: Shows the exact location of every ultra-fast charger without grouping, allowing you to see the precise distribution.
2. **Individual Points View and Road Segments**: Shows the exact location of every ultra-fast chargers against the road segments.

### 2.1 Clustered View


In [ ]:
# Clustered Visualization
center_lat, center_lon = df_sites['latitude'].mean(), df_sites['longitude'].mean()
m_clustered = folium.Map(location=[center_lat, center_lon], zoom_start=6, tiles="CartoDB Positron")
marker_cluster = MarkerCluster(name="Charging Points").add_to(m_clustered)

for row in df_sites.to_dicts():
    connectors_html = "".join([f"<li>{t} ({p} kW)</li>" for t, p in zip(row['connector_type'], row['max_power'])])
    popup_html = f"<b>{row['site_name']}</b><hr><ul>{connectors_html}</ul>"
    folium.Marker(
        [row['latitude'], row['longitude']],
        popup=folium.Popup(popup_html, max_width=300),
        tooltip=row['site_name']
    ).add_to(marker_cluster)

m_clustered

### 2.1 Individual Points View


In [ ]:
# Individual Points Visualization (No Clustering)
m_points = folium.Map(location=[center_lat, center_lon], zoom_start=6, tiles="CartoDB Positron")

for row in df_sites.to_dicts():
    # Use CircleMarker for better performance and cleaner look when not clustering
    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=3,
        color="#3186cc",
        fill=True,
        fill_color="#3186cc",
        fill_opacity=0.7,
        tooltip=row['site_name']
    ).add_to(m_points)

m_points

### 2.3 Total Ultra Fast Charging Points and Road Segments View

In [ ]:
# Individual Points Visualization (No Clustering)
m_roads_charging_points = folium.Map(location=[center_lat, center_lon], zoom_start=6, tiles="CartoDB Positron")

for row in df_sites.to_dicts():
    # Use CircleMarker for better performance and cleaner look when not clustering
    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=3,
        color="#3186cc",
        fill=True,
        fill_color="#3186cc",
        fill_opacity=0.7,
        tooltip=row['site_name']
    ).add_to(m_roads_charging_points)


# Add Processed Segments
folium.GeoJson(
    gdf_processed,
    name="Processed Network",
    style_function=lambda x: {'color': 'blue', 'weight': 2, 'opacity': 0.7},
    #tooltip=folium.GeoJsonTooltip(fields=["backbone_id", "original_segment_count"], aliases=["Backbone ID:", "Original Segments:"])
).add_to(m_roads_charging_points)

m_roads_charging_points

### 2.4 Ultra Fast Charging Points 1km or closer to Road Segments

In [ ]:
# Individual Points Visualization (No Clustering)
m_roads_charging_points = folium.Map(location=[center_lat, center_lon], zoom_start=6, tiles="CartoDB Positron")

for row in charging_stations.itertuples():
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=3,
        color="#3186cc",
        fill=True,
        fill_color="#3186cc",
        fill_opacity=0.7,
        tooltip=getattr(row, "site_name", "")
    ).add_to(m_roads_charging_points)


# Add Processed Segments
folium.GeoJson(
    gdf_processed,
    name="Processed Network",
    style_function=lambda x: {'color': 'blue', 'weight': 2, 'opacity': 0.7},
    #tooltip=folium.GeoJsonTooltip(fields=["backbone_id", "original_segment_count"], aliases=["Backbone ID:", "Original Segments:"])
).add_to(m_roads_charging_points)

m_roads_charging_points